In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from data_loading import *
from loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision set to 'medium'


In [2]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

y col is: sum_of_kWh, group col is: eic_code
features: ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'Month', 'Day', 'Hour', 'day_of_week', 'season', 'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast', 'dam_price', 'buy_bm_price', 'sell_bm_price', 'max_power', 'max_solar', 'max_ev']


In [ ]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …


In [ ]:
# # If a model cant handle categorical columns natively, or through embedings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [ ]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

In [ ]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

## PatchTST training

The cells below extend the template with a PyTorch PatchTST implementation. Design notes:

- **Channel-independent Transformer**: each feature (channel) is patched and passed through a *shared* encoder, then a per-channel linear head produces the forecast. This is the core PatchTST design from Nie et al. (2023).
- **Custom loss**: `money_pct` is implemented as a differentiable PyTorch op using the per-timestep price tensors (`dam_price`, `sell_bm_price`, `buy_bm_price`) which are looked up by `time_idx`.
- **Per-series target scaling**: each `eic_code` gets its own mean/std for `sum_of_kWh`. Exogenous features share global stats. Categoricals are integer-encoded fit on train and treated as numeric channels (channel-independence handles this without embeddings).
- **No leakage**: sliding windows are built per station and never cross train/val/test boundaries. Inference rolls forward `horizon` hours at a time using already-known future exogenous features.


In [ ]:
# PatchTST training stack — extra imports beyond the template
import math
import os
import random
import tempfile
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ---- PatchTST hyperparameters --------------------------------------------
@dataclass
class PatchTSTConfig:
    # windowing
    input_len:    int = 336      # 14 days lookback (14*24)
    horizon:      int = 48       # 48-hour forecast (matches market ordering window)
    patch_len:    int = 16       # patch size in timesteps
    stride:       int = 8        # overlap between patches
    # model
    d_model:      int = 128
    n_heads:      int = 8
    n_layers:     int = 3
    d_ff:         int = 256
    dropout:      float = 0.2
    # training
    batch_size:   int = 256
    epochs:       int = 15
    lr:           float = 5e-4
    weight_decay: float = 1e-5
    # loss
    loss_name:    str = "money_pct"   # "money_pct" | "mse" | "money_pct_smape"
    smape_weight: float = 0.1        # weight on smape term when using "money_pct_smape"
    # patience for early stopping (on val money_pct)
    patience:     int = 4
    # exogenous handling
    cat_cols: List[str] = field(default_factory=lambda: cat_columns.copy())
    num_cols: List[str] = field(default_factory=lambda: (
        weather_cols_all + ["latitude", "longitude",
                            "dam_price", "buy_bm_price", "sell_bm_price",
                            "max_power", "max_solar", "max_ev"]
    ))

CFG = PatchTSTConfig()
print(CFG)

# Number of patches the encoder will see per channel
N_PATCHES = (CFG.input_len - CFG.patch_len) // CFG.stride + 1
print(f"patches per channel: {N_PATCHES}")


In [ ]:
# ---- Feature preparation -------------------------------------------------
# Strategy:
#  * fit label encoders on train, apply to val/test (unseen categories -> -1)
#  * fit per-series target scaler (mean/std of sum_of_kWh per eic_code) on train
#  * fit global numeric scaler on train (mean/std per column)
#  * build a price lookup by time_idx for the money_pct loss
#  * concatenate frames into one continuous numeric matrix per station

ALL_FEATURES = CFG.num_cols + CFG.cat_cols   # ordering matters, keep it stable
PRICE_COLS = ["dam_price", "sell_bm_price", "buy_bm_price"]

# 1) Encode categoricals (fit on train only)
encoders: Dict[str, LabelEncoder] = {}
for col in CFG.cat_cols:
    le = LabelEncoder()
    le.fit(train[col].astype(str))
    encoders[col] = le

def encode_cats(df: pd.DataFrame) -> pd.DataFrame:
    out = df
    for col, le in encoders.items():
        known = set(le.classes_)
        vals = df[col].astype(str)
        # unseen categories -> -1 (handled as just another integer channel)
        mapped = np.where(vals.isin(known), vals, le.classes_[0])
        encoded = le.transform(mapped)
        encoded = np.where(vals.isin(known), encoded, -1)
        out = out.assign(**{col: encoded})
    return out

train = encode_cats(train)
val   = encode_cats(val)
test  = encode_cats(test)

# 2) Global numeric scaler on train (fit only on numeric exogenous features)
num_mean = train[CFG.num_cols].mean()
num_std  = train[CFG.num_cols].std().replace(0, 1.0)

def scale_num(df: pd.DataFrame) -> pd.DataFrame:
    df[CFG.num_cols] = (df[CFG.num_cols] - num_mean) / num_std
    return df

train = scale_num(train)
val   = scale_num(val)
test  = scale_num(test)

# 3) Per-series target scaler — only target column gets per-series stats
target_stats = (
    train.groupby(GROUP_COL)[Y_COL]
    .agg(["mean", "std"])
    .rename(columns={"mean": "y_mean", "std": "y_std"})
)
target_stats["y_std"] = target_stats["y_std"].replace(0, 1.0).fillna(1.0)

def add_scaled_target(df: pd.DataFrame) -> pd.DataFrame:
    stats = df[[GROUP_COL]].merge(target_stats, left_on=GROUP_COL, right_index=True, how="left")
    df["y_mean"] = stats["y_mean"].values
    df["y_std"]  = stats["y_std"].values
    df["y_scaled"] = (df[Y_COL] - df["y_mean"]) / df["y_std"]
    return df

train = add_scaled_target(train)
val   = add_scaled_target(val)
test  = add_scaled_target(test)

print("feature matrix columns:", len(ALL_FEATURES))
print("train range time_idx :", train["time_idx"].min(), "->", train["time_idx"].max())
print("val   range time_idx :", val["time_idx"].min(),   "->", val["time_idx"].max())
print("test  range time_idx :", test["time_idx"].min(),  "->", test["time_idx"].max())


In [ ]:
# ---- Price lookup keyed by time_idx --------------------------------------
# Prices depend only on time_idx (not on station), so we build one dense
# numpy array indexed by time_idx. This is what the money_pct loss reads.

all_prices = pd.concat([train, val, test])[["time_idx"] + PRICE_COLS].drop_duplicates("time_idx")
all_prices = all_prices.sort_values("time_idx").reset_index(drop=True)

min_idx = int(all_prices["time_idx"].min())
max_idx = int(all_prices["time_idx"].max())
PRICE_OFFSET = min_idx
n = max_idx - min_idx + 1
PRICE_TABLE = np.zeros((n, 3), dtype=np.float32)  # cols: dam, sell_bm, buy_bm

for c_i, col in enumerate(PRICE_COLS):
    PRICE_TABLE[all_prices["time_idx"].values - min_idx, c_i] = all_prices[col].values

PRICE_TABLE_T = torch.from_numpy(PRICE_TABLE).to(DEVICE)
print(f"price table shape : {PRICE_TABLE.shape}  (time_idx range {min_idx} -> {max_idx})")

def lookup_prices(time_idx_tensor: torch.Tensor) -> torch.Tensor:
    """time_idx_tensor: any shape, returns (..., 3) prices on DEVICE."""
    return PRICE_TABLE_T[time_idx_tensor - PRICE_OFFSET]


In [ ]:
# ---- Sliding-window dataset ----------------------------------------------
# Each sample:
#   x_enc   : [input_len, n_features]     scaled inputs (target uses y_scaled)
#   y_true  : [horizon]                   target in ORIGINAL units (kWh)
#   y_mean,
#   y_std   : scalars                     per-series stats to invert scaling
#   tidx    : [horizon]                   time_idx of horizon (for prices)
# Sliding windows are built per station and never cross station boundaries.

# index of y_scaled within the feature matrix
TARGET_FEAT_IDX = ALL_FEATURES.index("dam_price")  # placeholder, overwritten below
# Actually we want a dedicated channel for the *scaled target* in x_enc, so we
# add it explicitly as the FIRST channel — channel 0 is always the target.
FEATURE_COLS = ["y_scaled"] + ALL_FEATURES
N_CHANNELS = len(FEATURE_COLS)
TARGET_CH = 0
print(f"channels (incl. target): {N_CHANNELS}")

def build_windows(df: pd.DataFrame, cfg: PatchTSTConfig, stride: int = 1):
    """Yield (station_arr, y_true, y_mean, y_std, tidx) per sliding window.

    `stride` controls window step at training time (1 = dense, larger = faster).
    """
    L, H = cfg.input_len, cfg.horizon
    samples = []
    df = df.sort_values([GROUP_COL, "time_idx"])
    for grp, gdf in df.groupby(GROUP_COL, sort=False):
        # ensure contiguous time_idx — your data already is
        feat = gdf[FEATURE_COLS].to_numpy(dtype=np.float32)
        y_raw = gdf[Y_COL].to_numpy(dtype=np.float32)
        tidx  = gdf["time_idx"].to_numpy(dtype=np.int64)
        y_mean = float(gdf["y_mean"].iloc[0])
        y_std  = float(gdf["y_std"].iloc[0])
        n_t = len(gdf)
        last_start = n_t - L - H
        if last_start < 0:
            continue
        for s in range(0, last_start + 1, stride):
            samples.append((
                feat[s:s+L],                # [L, C]
                y_raw[s+L : s+L+H],         # [H] in kWh
                y_mean, y_std,
                tidx[s+L : s+L+H],          # [H]
            ))
    return samples


class WindowDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        feat, y_true, y_mean, y_std, tidx = self.samples[i]
        return (
            torch.from_numpy(feat),
            torch.from_numpy(y_true),
            torch.tensor(y_mean, dtype=torch.float32),
            torch.tensor(y_std,  dtype=torch.float32),
            torch.from_numpy(tidx),
        )


# Train: use stride=H to get non-overlapping samples (faster + still plenty)
TRAIN_STRIDE = CFG.horizon
print("building training windows ...")
train_samples = build_windows(train, CFG, stride=TRAIN_STRIDE)
print(f"  train windows : {len(train_samples):,}")

# Val: dense (stride=1) is small (one month) and we want a tight signal
print("building validation windows ...")
val_samples = build_windows(val, CFG, stride=CFG.horizon)
print(f"  val windows   : {len(val_samples):,}")

train_ds = WindowDataset(train_samples)
val_ds   = WindowDataset(val_samples)

train_loader = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG.batch_size, shuffle=False,
    num_workers=0, pin_memory=True,
)
print(f"train batches : {len(train_loader)}")
print(f"val batches   : {len(val_loader)}")


In [ ]:
# ---- Differentiable money_pct loss ---------------------------------------
# Mirrors the rule in loss_funcs.money_pct:
#   * if pred is within +/-10% of true: settled at dam_price
#   * if over-prediction beyond 10%: extra kWh sold back at sell_bm_price
#                                    (typically lower -> a loss vs dam_price)
#   * if under-prediction beyond 10%: missing kWh bought at buy_bm_price
#                                    (typically higher -> a loss vs dam_price)
# Result is normalized by the cost of perfect prediction at dam_price -> %.

def money_pct_torch(
    y_true: torch.Tensor,        # [B, H]   kWh
    y_pred: torch.Tensor,        # [B, H]   kWh
    dam:    torch.Tensor,        # [B, H]
    sell:   torch.Tensor,        # [B, H]
    buy:    torch.Tensor,        # [B, H]
    band:   float = 0.10,
    eps:    float = 1e-6,
) -> torch.Tensor:
    upper = y_true * (1.0 + band)
    lower = y_true * (1.0 - band)

    # how far outside the band the prediction lies (0 inside)
    over  = torch.relu(y_pred - upper)   # excess kWh ordered, must be sold
    under = torch.relu(lower - y_pred)   # missing kWh, must be bought

    # base settlement is at dam_price for the actual delivered amount
    perfect_cost = y_true * dam

    # actual cost vs perfect:
    #   over  : we paid `over` kWh at dam but only get sell back -> loss = over*(dam - sell)
    #   under : we lacked `under` kWh, must buy at buy           -> loss = under*(buy - dam)
    loss_amount = over * (dam - sell) + under * (buy - dam)

    total_perfect = perfect_cost.abs().sum() + eps
    return 100.0 * loss_amount.sum() / total_perfect


def smape_torch(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Differentiable SMAPE in percent (0-200 scale).

    Used as a dense regularizer on top of money_pct, which is sparse — it has
    zero gradient inside the +/-10% band and zero gradient when over- or
    under-prediction is exactly proportional to dam-sell or buy-dam. SMAPE
    keeps a non-zero gradient everywhere predictions disagree with truth.
    """
    num = torch.abs(y_pred - y_true)
    den = (torch.abs(y_true) + torch.abs(y_pred)) / 2.0 + eps
    return 100.0 * torch.mean(num / den)


def composite_loss(
    y_true: torch.Tensor, y_pred: torch.Tensor,
    dam: torch.Tensor, sell: torch.Tensor, buy: torch.Tensor,
    smape_weight: float,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Returns (total_loss, money_pct_term, smape_term) — all in % units."""
    mp = money_pct_torch(y_true, y_pred, dam, sell, buy)
    sm = smape_torch(y_true, y_pred)
    return mp + smape_weight * sm, mp, sm


def mse_loss_fn(y_true, y_pred, *_args, **_kw):
    return torch.mean((y_true - y_pred) ** 2)


# ---- Sanity: differentiable money_pct must match loss_funcs.money_pct -----
# Quick numerical agreement check on synthetic data — protects against silent
# drift if either implementation changes later.
def _check_money_pct_agreement():
    rng = np.random.default_rng(0)
    B, H = 32, 48
    y = rng.uniform(50, 200, (B, H)).astype(np.float32)
    p = (y + rng.normal(0, 30, (B, H))).astype(np.float32)
    dam_  = rng.uniform(80, 120, (B, H)).astype(np.float32)
    sell_ = (dam_ * rng.uniform(0.5, 1.0, (B, H))).astype(np.float32)
    buy_  = (dam_ * rng.uniform(1.0, 1.5, (B, H))).astype(np.float32)

    ref = money_pct(y.flatten(), p.flatten(), dam_.flatten(), sell_.flatten(), buy_.flatten())
    impl = money_pct_torch(
        torch.from_numpy(y), torch.from_numpy(p),
        torch.from_numpy(dam_), torch.from_numpy(sell_), torch.from_numpy(buy_),
    ).item()
    print(f"money_pct (loss_funcs)        : {ref:.6f}")
    print(f"money_pct_torch (this module) : {impl:.6f}")
    assert abs(ref - impl) < 1e-3, f"money_pct implementations disagree: {ref} vs {impl}"
    print("[ok] money_pct_torch matches loss_funcs.money_pct")

_check_money_pct_agreement()


In [ ]:
# ---- PatchTST model (channel-independent) --------------------------------
# Channel-independent design: every channel is unfolded into patches and passed
# through the SAME Transformer encoder. The forecast for the target channel
# (channel 0) is produced by a flatten-and-linear head on its own encoded
# tokens. Other channels still flow through the shared encoder so their weights
# get gradient signal — but only the target channel's head produces the output.
# This is the standard PatchTST channel-independence formulation.

class PatchTST(nn.Module):
    def __init__(self, cfg: PatchTSTConfig, n_channels: int):
        super().__init__()
        self.cfg = cfg
        self.n_channels = n_channels
        self.n_patches = (cfg.input_len - cfg.patch_len) // cfg.stride + 1

        # patch embedding (shared across channels)
        self.patch_proj = nn.Linear(cfg.patch_len, cfg.d_model)

        # learnable positional encoding over patches
        self.pos_emb = nn.Parameter(torch.randn(1, self.n_patches, cfg.d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_ff,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg.n_layers)

        # forecasting head: flatten patch tokens for the TARGET channel -> horizon
        self.dropout = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(self.n_patches * cfg.d_model, cfg.horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [B, L, C]
        returns y_hat (scaled) : [B, H]  for the target channel only
        """
        B, L, C = x.shape
        # Channel-first: [B, C, L]
        x = x.transpose(1, 2)
        # Patch via unfold: [B, C, n_patches, patch_len]
        x = x.unfold(dimension=-1, size=self.cfg.patch_len, step=self.cfg.stride)
        # merge B and C so the encoder sees each channel independently with shared weights
        x = x.contiguous().view(B * C, self.n_patches, self.cfg.patch_len)
        # embed + positional
        x = self.patch_proj(x) + self.pos_emb       # [B*C, n_patches, d_model]
        # encode
        z = self.encoder(x)                          # [B*C, n_patches, d_model]
        # split back: [B, C, n_patches * d_model]
        z = z.reshape(B, C, self.n_patches * self.cfg.d_model)
        # take only the target channel's representation
        z_tgt = self.dropout(z[:, TARGET_CH, :])     # [B, n_patches * d_model]
        return self.head(z_tgt)                      # [B, H]


In [ ]:
# ---- Training loop -------------------------------------------------------
mlflow.set_experiment("patchtst_electricity")

def evaluate_loader(model, loader):
    """Return (money_pct%, smape%) averaged over the loader, no grad."""
    model.eval()
    mps, sms = [], []
    with torch.no_grad():
        for x, y_true_kwh, y_mean, y_std, tidx in loader:
            x          = x.to(DEVICE, non_blocking=True)
            y_true_kwh = y_true_kwh.to(DEVICE, non_blocking=True)
            y_mean     = y_mean.to(DEVICE).unsqueeze(1)
            y_std      = y_std.to(DEVICE).unsqueeze(1)
            tidx       = tidx.to(DEVICE)

            y_hat_scaled = model(x)
            y_hat_kwh    = y_hat_scaled * y_std + y_mean

            prices = lookup_prices(tidx)             # [B, H, 3]
            dam, sell, buy = prices[..., 0], prices[..., 1], prices[..., 2]

            mps.append(money_pct_torch(y_true_kwh, y_hat_kwh, dam, sell, buy).item())
            sms.append(smape_torch(y_true_kwh, y_hat_kwh).item())
    return float(np.mean(mps)), float(np.mean(sms))


model = PatchTST(CFG, n_channels=N_CHANNELS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"PatchTST parameters: {n_params:,}")

optim = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=CFG.epochs)

best_state = None
best_val_mp = float("inf")
patience_left = CFG.patience

mlflow.start_run(run_name="patchtst")

mlflow.log_params({
    "model": "PatchTST",
    "input_len":   CFG.input_len,
    "horizon":     CFG.horizon,
    "patch_len":   CFG.patch_len,
    "stride":      CFG.stride,
    "n_patches":   N_PATCHES,
    "d_model":     CFG.d_model,
    "n_heads":     CFG.n_heads,
    "n_layers":    CFG.n_layers,
    "d_ff":        CFG.d_ff,
    "dropout":     CFG.dropout,
    "batch_size":  CFG.batch_size,
    "epochs":      CFG.epochs,
    "lr":          CFG.lr,
    "weight_decay":CFG.weight_decay,
    "loss_name":   CFG.loss_name,
    "smape_weight":CFG.smape_weight,
    "n_channels":  N_CHANNELS,
    "n_features":  len(FEATURE_COLS),
    "n_train_windows": len(train_samples),
    "n_val_windows":   len(val_samples),
    "train_stations":  train[GROUP_COL].nunique(),
    "train_date_min":  str(train["datetime"].min()),
    "train_date_max":  str(train["datetime"].max()),
    "val_date_min":    str(val["datetime"].min()),
    "val_date_max":    str(val["datetime"].max()),
    "test_date_min":   str(test["datetime"].min()),
    "test_date_max":   str(test["datetime"].max()),
    "device":          str(DEVICE),
})

# log feature columns as a small artifact
with tempfile.TemporaryDirectory() as td:
    f = Path(td) / "features.txt"
    f.write_text("\n".join(FEATURE_COLS))
    mlflow.log_artifact(str(f))

epoch_bar = tqdm(range(1, CFG.epochs + 1), desc="epochs")
for epoch in epoch_bar:
    model.train()
    train_mp_running, train_sm_running, n_seen = 0.0, 0.0, 0

    batch_bar = tqdm(train_loader, desc=f"epoch {epoch}", leave=False)
    for x, y_true_kwh, y_mean, y_std, tidx in batch_bar:
        x          = x.to(DEVICE, non_blocking=True)
        y_true_kwh = y_true_kwh.to(DEVICE, non_blocking=True)
        y_mean_b   = y_mean.to(DEVICE).unsqueeze(1)
        y_std_b    = y_std.to(DEVICE).unsqueeze(1)
        tidx       = tidx.to(DEVICE)

        y_hat_scaled = model(x)
        y_hat_kwh    = y_hat_scaled * y_std_b + y_mean_b

        prices = lookup_prices(tidx)
        dam, sell, buy = prices[..., 0], prices[..., 1], prices[..., 2]

        # Always compute both terms so we can report them, regardless of which
        # loss is being optimized.
        mp_term = money_pct_torch(y_true_kwh, y_hat_kwh, dam, sell, buy)
        sm_term = smape_torch(y_true_kwh, y_hat_kwh)

        if CFG.loss_name == "money_pct":
            loss = mp_term
        elif CFG.loss_name == "money_pct_smape":
            loss = mp_term + CFG.smape_weight * sm_term
        elif CFG.loss_name == "mse":
            y_true_scaled = (y_true_kwh - y_mean_b) / y_std_b
            loss = mse_loss_fn(y_true_scaled, y_hat_scaled)
        else:
            raise ValueError(f"unknown loss_name: {CFG.loss_name}")

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optim.step()

        bs = x.size(0)
        train_mp_running += mp_term.item() * bs
        train_sm_running += sm_term.item() * bs
        n_seen += bs
        batch_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            mp=f"{mp_term.item():.3f}",
            sm=f"{sm_term.item():.3f}",
        )

    sched.step()
    train_pct   = train_mp_running / n_seen
    train_smape = train_sm_running / n_seen
    val_pct, val_smape = evaluate_loader(model, val_loader)

    print(f"epoch {epoch:2d} | "
          f"train_pct={train_pct:.3f}% train_smape={train_smape:.3f} "
          f"val_pct={val_pct:.3f}% val_smape={val_smape:.3f}")

    mlflow.log_metrics({
        "train_money_pct": train_pct,
        "train_smape":     train_smape,
        "val_money_pct":   val_pct,
        "val_smape":       val_smape,
        "lr":              optim.param_groups[0]["lr"],
    }, step=epoch)
    epoch_bar.set_postfix(
        train_pct=f"{train_pct:.3f}",
        val_pct=f"{val_pct:.3f}",
        val_smape=f"{val_smape:.3f}",
    )

    if val_pct < best_val_mp - 1e-6:
        best_val_mp = val_pct
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = CFG.patience
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch} — best val_money_pct={best_val_mp:.4f}")
            break

# restore best weights and save artifact
if best_state is not None:
    model.load_state_dict(best_state)
ckpt_path = Path(tempfile.gettempdir()) / "patchtst_best.pt"
torch.save({"state_dict": model.state_dict(), "config": CFG.__dict__}, ckpt_path)
mlflow.log_artifact(str(ckpt_path))
print(f"best val_money_pct (window-level): {best_val_mp:.4f}")


In [ ]:
# ---- Inference: rolling 48h forecasts to cover the full eval window ------
# We have ~1 month of val/test data per station and a 48h horizon. We roll
# forward in `horizon`-sized chunks. The encoder context for each chunk uses
# the *true* observed history up to that point (not predictions) — this is
# the standard "rolling-origin" eval and avoids compounding error.
# We need `input_len` hours of context immediately preceding each chunk;
# train tail provides the warm-up for val, and (train+val) provides it for test.

def run_rolling_inference(eval_df: pd.DataFrame, history_df: pd.DataFrame, model, cfg: PatchTSTConfig) -> pd.DataFrame:
    """
    eval_df    : the slice we want predictions for (val or test)
    history_df : everything BEFORE eval_df chronologically, per station
                 (used to seed the input context window). Must contain the
                 same engineered/scaled columns as eval_df.
    Returns eval_df with a 'pred' column added (predictions in original kWh).
    """
    model.eval()
    L, H = cfg.input_len, cfg.horizon

    eval_df = eval_df.sort_values([GROUP_COL, "time_idx"])
    history_df = history_df.sort_values([GROUP_COL, "time_idx"])

    preds_all: List[pd.DataFrame] = []
    stations = eval_df[GROUP_COL].unique()

    for grp in tqdm(stations, desc="rolling inference"):
        e = eval_df[eval_df[GROUP_COL] == grp]
        if len(e) == 0:
            continue
        h = history_df[history_df[GROUP_COL] == grp].tail(L)
        if len(h) < L:
            # not enough history — skip this station
            continue

        # full per-station feature trace (history tail + eval period)
        full = pd.concat([h, e], ignore_index=True)
        feat_full = full[FEATURE_COLS].to_numpy(dtype=np.float32)
        tidx_full = full["time_idx"].to_numpy(dtype=np.int64)
        y_mean = float(e["y_mean"].iloc[0])
        y_std  = float(e["y_std"].iloc[0])

        n_eval = len(e)
        preds_kwh = np.zeros(n_eval, dtype=np.float32)

        # we always read context as feat_full[start : start+L] where start
        # advances by H each step. start=0 -> first context is exactly `h`
        # (the L hours just before eval starts).
        with torch.no_grad():
            cursor = 0
            while cursor < n_eval:
                ctx = feat_full[cursor : cursor + L]                    # [L, C]
                if len(ctx) < L:
                    break
                x = torch.from_numpy(ctx).unsqueeze(0).to(DEVICE)        # [1, L, C]
                y_hat_scaled = model(x).squeeze(0).cpu().numpy()         # [H]
                y_hat_kwh = y_hat_scaled * y_std + y_mean

                end = min(cursor + H, n_eval)
                take = end - cursor
                preds_kwh[cursor : end] = y_hat_kwh[:take]
                cursor += H

        out = e.copy()
        out["pred"] = preds_kwh
        preds_all.append(out)

    return pd.concat(preds_all, ignore_index=True) if preds_all else pd.DataFrame()


# Build the eval frames. For val, history = train. For test, history = train + val.
print("running rolling inference on validation ...")
val_eval = run_rolling_inference(val, train, model, CFG)

print("running rolling inference on test ...")
test_eval = run_rolling_inference(test, pd.concat([train, val], ignore_index=True), model, CFG)

# Restore original kWh in y_true column for evaluation cells (val/test
# already have Y_COL untouched in the original units — feature scaling did
# not modify Y_COL itself, only the y_scaled helper column).
print(f"val_eval rows  : {len(val_eval):,}")
print(f"test_eval rows : {len(test_eval):,}")
print(val_eval[[GROUP_COL, "time_idx", "datetime", Y_COL, "pred", "dam_price"]].head())


In [ ]:
def _bias(y, p): return float(np.sum(np.asarray(p)) - np.sum(np.asarray(y)))

def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_bias_v      = _bias(val_eval[Y_COL], val_eval['pred'])

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_bias_v      = _bias(test_eval[Y_COL], test_eval['pred'])

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")
print(f"BIAS      : {val_bias_v:.4f}")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")
print(f"BIAS      : {test_bias_v:.4f}")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "val_bias":       val_bias_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
    "test_bias":      test_bias_v,
})
mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money    (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct(gdf[Y_COL], gdf["pred"], *_prices(gdf)),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")


test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")